<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento del Lenguaje Natural*

#### Profesor: Lucas Iturriago

# Taller 1 — Duelo secuencial en SMS Spam: RNN vs. LSTM/GRU vs. Transformers

**Unidad 2 (Clases 5–9). Dificultad: media-alta. Individual. Solo conceptos vistos en clase.**

Una operadora móvil quiere un filtro de SMS spam en español-inglés corto (jerga, mayúsculas, desbalance). Usted debe demostrar, con el **mismo dataset `SMSSpamCollection`** de las Clases 5 y 7, que el orden importa: comparar modelos recurrentes contra Transformers encoder-only/decoder-only, con protocolo común y análisis de robustez.

**Dataset:** `Insumos/SMSSpamCollection.txt` (formato `label\tmessage`, 5574 filas, ~13% spam) o URL:
`https://raw.githubusercontent.com/UN-GCPDS/ProcesamientoLenguajeNatural/refs/heads/main/Insumos/SMSSpamCollection.txt`

**Protocolo obligatorio (para que la comparación sea justa):**
- `train_test_split(test_size=0.2, stratify=y, random_state=42)`.
- Tokenización: minúsculas + `re.sub(r"[^\\w\\s]", "", ...)` + `split()` (la de Clase 5).
- Vocabulario con `[PAD]=0, [UNK]=1`.
- `pad_len` = percentil 95 de longitudes en train (aceptado 28–35).
- Recurrentes: `embedding_dim=64, hidden_dim=128`, `BCEWithLogitsLoss`, `Adam(lr=1e-3)`, 5 épocas.
- Transformers: `d_model=128, heads=4, layers=2-3, d_ff=512, dropout=0.1`, `Adam(lr=1e-4)`, `BCELoss/BCEWithLogits`, 3 épocas.
- Todo en CPU es suficiente (GPU opcional, sin ventaja). Tiempo esperado total: ~5–10 min. Si una época tarda >3 min, revise que esté usando `pad_len` p95 (≈32) y los hiperparámetros del protocolo. Fije semillas (`torch.manual_seed(42)`, `np.random.seed(42)`).
- **Restricción:** prohibido fine-tuning de BERT/GPT-2 externos o embeddings de la Clase 9 con API. Solo PyTorch desde cero como en Clases 5–7, más `sklearn` para métricas.


---
## E1 — Pipeline: vocab + padding p95 + DataLoader (vale 1.0)

**Haga:**
1. Codifique `ham→0, spam→1`, reporte desbalance (`y_train_mean`).
2. Construya vocabulario con frecuencia >1 y especiales `[PAD]/[UNK]`.
3. Calcule `pad_len` como percentil 95 de `msg_len` en train.
4. Implemente `encode_and_pad(text, vocab, pad_len)` (truncar/padear, `[UNK]` para OOV).
5. Cree `SpamDataset/SMSDataset` y `train_loader/test_loader` (`batch_size=64`, shuffle solo en train).

**Variables a entregar:** `vocab_size` (int), `pad_len` (int), `X_train_shape` (tuple), `y_train_mean` (float).

**Chequeo automático:** `28 <= pad_len <= 35`, `3500 <= vocab_size <= 9500`, `X_train_shape[0] ≈ 4459`, `0.10 <= y_train_mean <= 0.17`.


---
## E2 — Recurrentes: SimpleRNN vs. LSTM vs. GRU (vale 1.0)

**Haga:**
1. Implemente `SimpleRNNModel` y `LSTMModel` como en Clase 5.
2. Implemente `GRUModel` con `nn.GRU` (recuerde: la GRU **no** tiene `cell_state`, solo `hidden_state`).
3. Entrene los 3 con el loop manual (`zero_grad → forward → loss → backward → step`), 5 épocas, y guarde la pérdida por época.
4. Evalúe `accuracy` en test con `torch.no_grad()` + `sigmoid > 0.5`.

**Variables a entregar:** `rnn_acc`, `lstm_acc`, `gru_acc` (floats 0–1), `e2_train_losses` (dict `{"rnn": [...5], "lstm": [...5], "gru": [...5]}`).

**Chequeo automático:** cada lista tiene 5 valores decrecientes en tendencia (última < primera), `0.70 <= acc <= 1.0`. Se espera `lstm_acc, gru_acc >= rnn_acc - 0.03` (las compuertas olvidan menos).


---
## E3 — Transformers: Encoder-only vs. Decoder-only (vale 1.0)

**Haga:**
1. Implemente `EncoderOnlyTransformer` (self-attention bidireccional + pooling promedio enmascarando `[PAD]`, como Clase 7).
2. Implemente `DecoderOnlyTransformer` (máscara causal triangular + pooling del último token válido, como Clase 7).
3. Entrene ambos 3 épocas con el mismo split y reporte `accuracy` y `AUC` (`sklearn.metrics.accuracy_score, roc_auc_score`).

**Variables a entregar:** `enc_acc`, `enc_auc`, `dec_acc`, `dec_auc` (floats).

**Chequeo automático:** `0.80 <= acc <= 1.0`, `0.85 <= auc <= 1.0`. En SMS cortos se espera `enc_acc >= dec_acc - 0.03` (el encoder bidireccional aprovecha contexto derecho; el decoder sufre por posición).


---
## E4 — Robustez: ablación + orden + longitud (vale 1.0)

**Haga:**
1. **Ablación PE:** entrene el Encoder-only **sin** `PositionalEncoding` y guarde `enc_nopos_acc`. Explique en 2 líneas por qué cae o no.
2. **Test de orden:** tome 200 SMS del test, permute aleatoriamente las palabras de cada uno (`random.seed(42)`), evalúe su mejor modelo y guarde `acc_shuffled` y `acc_drop = acc_orig - acc_shuffled`.
3. **Por longitud:** con su mejor modelo, calcule accuracy en 3 buckets del test: `corto (len<=15)`, `medio (16..pad_len)`, `largo (>pad_len, truncados)`. Guarde `acc_by_length = {"corto":…, "medio":…, "largo":…}`.

**Variables a entregar:** `enc_nopos_acc`, `acc_shuffled`, `acc_drop`, `acc_by_length` (dict con las 3 claves).

**Chequeo automático:** `acc_drop > 0.02` (si el modelo usa el orden, permutar lo rompe; un BoW no caería). `acc_by_length["largo"] <= acc_by_length["corto"] + 0.05` (el truncado p95 penaliza largos). `enc_nopos_acc <= enc_acc + 0.02`.


---
## E5 — XAI con atención + decisión de despliegue (vale 1.0)

**Haga:**
1. Extraiga los pesos de atención de la **última capa** del Encoder-only (promedio de las 4 cabezas) para 4 mensajes del test: 2 aciertos y 2 errores. Grafique 1 heatmap (tokens × tokens) con la máscara de `[PAD]` aplicada.
2. Guarde `attn_weights_shape` (ej. `(4, L, L)` o `(4, heads, L, L)`) y `top_tokens` (dict `{idx: [tok1, tok2, tok3]}` con los 3 tokens de mayor atención promedio por mensaje).
3. Escriba `decision_text` (5–8 líneas): ¿qué modelo desplegaría en la operadora y por qué? Vincule a: orden vs. BoW (E4), máscaras (E3), costo (`#params`/tiempo por época) y errores observados.

**Variables a entregar:** `attn_weights_shape` (tuple), `top_tokens` (dict de 4 entradas), `decision_text` (str).

**Chequeo automático:** `attn_weights_shape` coherente con `pad_len`, atención sobre `[PAD]` ≈ 0, `top_tokens` con 4 entradas no vacías, `decision_text` ≥ 60 palabras y menciona al menos 2 de: orden, máscara/padding, overfitting, costo/latencia.


---
## Entrega y calificación

- Entregue **un solo `.ipynb`** que corra de arriba a abajo en Colab (CPU). Todas las variables del cuadro deben existir en el namespace con esos nombres exactos.
- Incluya las curvas de `loss` por época y la tabla comparativa final (`rnn/lstm/gru/enc/dec` con `acc` y `auc`).
- Escala UNAL **0.0–5.0, un decimal, aprueba ≥ 3.0**. Cada E vale 1.0 y la final es la **suma**. Ver `Taller 1 - Rubrica IA.md`.

| Variable | Dónde | Tipo |
|---|---|---|
| `vocab_size, pad_len, X_train_shape, y_train_mean` | E1 | int, int, tuple, float |
| `rnn_acc, lstm_acc, gru_acc, e2_train_losses` | E2 | float×3, dict |
| `enc_acc, enc_auc, dec_acc, dec_auc` | E3 | float×4 |
| `enc_nopos_acc, acc_shuffled, acc_drop, acc_by_length` | E4 | float, float, float, dict |
| `attn_weights_shape, top_tokens, decision_text` | E5 | tuple, dict, str |

> Nota: los umbrales numéricos de esta guía son de plausibilidad (formato/rango), no de ranking. La nota la asigna la rúbrica IA contra lo entregado.
